# Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
# from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

#Importaciones faltantes
from sklearn.linear_model import LinearRegression, LogisticRegression, ridge_regression
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Cargar los datos

In [72]:
data = pd.read_csv('./data/Etapa1DatosEstudiantes.csv' , delimiter=';')

# Análisis exploratorio de los datos y Calidad de datos

Copiamos los datos como buena práctica

In [73]:
df = data.copy()

Analizamos el número de datos por columna y el tipo

In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1161 entries, 0 to 1160
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Id                          1161 non-null   int64  
 1   Sexo                        1161 non-null   str    
 2   Edad                        1160 non-null   float64
 3   MiembrosFamilia             1160 non-null   float64
 4   ViveConPadres               1142 non-null   str    
 5   TrabajoMama                 1159 non-null   str    
 6   TrabajoPapa                 1159 non-null   str    
 7   Cuidador                    1158 non-null   str    
 8   TiempoDedicadoEstudioNum    1158 non-null   float64
 9   ApoyoEscolar                1153 non-null   str    
 10  ApoyoEscolarFamilia         1146 non-null   str    
 11  QuiereEstudioUniversitario  1159 non-null   str    
 12  RelacionRomantica           1159 non-null   str    
 13  CalidadRelFamilia           1159 non-null   

In [75]:
df.describe(include='all')

,Id,Sexo,Edad,MiembrosFamilia,ViveConPadres,TrabajoMama,TrabajoPapa,Cuidador,TiempoDedicadoEstudioNum,ApoyoEscolar,ApoyoEscolarFamilia,QuiereEstudioUniversitario,RelacionRomantica,CalidadRelFamilia,TiempoLibre,SaleConAmigos,Ausencias,CantAlcPorSem,PromNotas
count,1161.000000,1161,1160.000000,1160.000000,1142,1159,1159,1158,1158.000000,1153,1146,1159,1159,1159.000000,1159.000000,1159.000000,1159.000000,1153.000000,1161.000000
unique,NaN,2,NaN,NaN,2,5,5,10,NaN,3,5,3,3,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,F,NaN,NaN,S,Otros,Otros,Mama,NaN,no,Si,Si,Si,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,645,NaN,NaN,821,356,469,601,NaN,577,651,849,477,NaN,NaN,NaN,NaN,NaN,NaN
mean,571.674419,NaN,16.671552,3.551724,NaN,NaN,NaN,NaN,4.591537,NaN,NaN,NaN,NaN,4.095772,3.827437,3.108714,7.188093,2.307025,11.244866
std,333.244669,NaN,1.774097,1.384898,NaN,NaN,NaN,NaN,4.333125,NaN,NaN,NaN,NaN,15.735848,13.956059,1.276296,11.264407,1.203768,3.245180
min,1.000000,NaN,13.000000,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,-4.000000,-3.000000,1.000000,0.000000,1.000000,1.330000
25%,282.000000,NaN,16.000000,2.000000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,3.000000,2.000000,2.000000,1.000000,1.000000,9.330000
50%,572.000000,NaN,17.000000,4.000000,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,NaN,NaN,4.000000,3.000000,3.000000,5.000000,2.000000,11.330000
75%,859.000000,NaN,18.000000,5.000000,NaN,NaN,NaN,NaN,7.000000,NaN,NaN,NaN,NaN,4.000000,4.000000,4.000000,11.000000,3.000000,13.330000


El análisis exploratorio evidencia algunas inconsistencias y problemas de calidad en los datos, como valores fuera de rango en variables ordinales (por ejemplo, CalidadRelFamilia y TiempoLibre presentan valores negativos y extremadamente altos) y posibles outliers en Ausencias. Además, se observan valores faltantes en varias variables categóricas y numéricas, lo que requiere procesos de limpieza, validación de rangos e imputación antes de la construcción de los modelos.

## Limpieza previa

In [76]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

In [77]:
for col in categorical_cols:
    print(col + ":")
    display(df[col].unique())

Sexo:


<StringArray>
['F', 'M']
Length: 2, dtype: str

ViveConPadres:


<StringArray>
['S', 'N', nan]
Length: 3, dtype: str

TrabajoMama:


...

<StringArray>
['Si', 'No', nan, 'no', 's ', 'n ']
Length: 6, dtype: str

QuiereEstudioUniversitario:


<StringArray>
['Si', 'No', nan, 'no']
Length: 4, dtype: str

RelacionRomantica:


<StringArray>
['Si', 'No', nan, 'no']
Length: 4, dtype: str

In [78]:
df = df.fillna("desconocido")
for columna in categorical_cols:
    if columna != 'Sexo':
        df[columna] = (
            df[columna]
            .str.lower()
            .str.strip()
            .replace({
                's': 'si',
                'n': 'no',
                'pa': 'papa',
                'ma': 'mama',
                'p': 'papa',
                'm': 'mama'
            })
        )
    else:
        df[columna] = (
            df[columna]
            .str.lower()
            .str.strip()
            .replace({
                's': 'si',
                'n': 'no',
                'pa': 'papa',
                'ma': 'mama',
                'p': 'papa'
            })
        )

In [79]:
for col in categorical_cols:
    display(df[col].unique())

<StringArray>
['f', 'm']
Length: 2, dtype: str

<StringArray>
['si', 'no', 'desconocido']
Length: 3, dtype: str

<StringArray>
['educacion', 'otros', 'desconocido', 'am@decasa', 'salud', 'servicios']
Length: 6, dtype: str

<StringArray>
['am@decasa', 'otros', 'desconocido', 'salud', 'servicios', 'educacion']
Length: 6, dtype: str

<StringArray>
['mama', 'papa', 'otros', 'desconocido']
Length: 4, dtype: str

<StringArray>
['si', 'no', 'desconocido']
Length: 3, dtype: str

<StringArray>
['si', 'no', 'desconocido']
Length: 3, dtype: str

<StringArray>
['si', 'no', 'desconocido']
Length: 3, dtype: str

<StringArray>
['si', 'no', 'desconocido']
Length: 3, dtype: str